In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import*
from pyspark.sql.functions import*


In [4]:
#Define the schema (Dataframe)
schema = StructType([
    StructField("Order_Date", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Quantity", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Profit", DoubleType(), True)
])

spark = SparkSession.builder.appName("BigData").getOrCreate()
path = "/content/sample_data/ecommerce_sales_data_unprocessed.csv"
df = spark.read.csv(path, header=True, schema=schema)

df.printSchema()
df.show()

root
 |-- Order_Date: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Profit: double (nullable = true)

+----------+----------+-----------+------+--------+------+-------+
|Order_Date|      Name|   Category|Region|Quantity| Sales| Profit|
+----------+----------+-----------+------+--------+------+-------+
|12/31/2024|   Printer|     Office| North|       4|3640.0| 348.93|
|12/31/2024|   Printer|     Office| North|       4|3640.0| 348.93|
|11/27/2022|     Mouse|Accessories|  East|       0|1197.0| 106.53|
| 5/11/2022|    Tablet|Electronics| South|       5|5865.0| 502.73|
| 3/16/2024|     Mouse|Accessories| South|       2| 786.0| 202.87|
| 9/10/2022|     Mouse|Accessories|  West|       1| 509.0| 103.28|
| 9/10/2022|     Mouse|Accessories|  West|       1| 509.0| 103.28|
| 12/1/2023|    Camera|Electronics|  West|       1|  

In [5]:
#cleaning for duplicates
columns = df.columns
duplicates = df.groupBy(columns).count().filter(F.col("count") > 1).drop("count")
print("Duplicates")
duplicates.show()

print("Cleaned DataFrame Sample:")
df_clean1 = df.dropDuplicates().orderBy("Order_Date")
df_clean1.show()
print("Before Cleaning Count: ", df.count())
print("After Cleaning Count: ", df_clean1.count())


Duplicates
+----------+-------+-----------+------+--------+------+------+
|Order_Date|   Name|   Category|Region|Quantity| Sales|Profit|
+----------+-------+-----------+------+--------+------+------+
| 9/10/2022|  Mouse|Accessories|  West|       1| 509.0|103.28|
|12/31/2024|Printer|     Office| North|       4|3640.0|348.93|
+----------+-------+-----------+------+--------+------+------+

Cleaned DataFrame Sample:
+----------+----------+-----------+------+--------+------+-------+
|Order_Date|      Name|   Category|Region|Quantity| Sales| Profit|
+----------+----------+-----------+------+--------+------+-------+
|  1/1/2022|     Mouse|Accessories|  East|       3|3474.0|  565.0|
|  1/1/2022|Headphones|Accessories| South|       7|3941.0| 628.96|
|  1/1/2022|   Monitor|Accessories| South|       8|1296.0| 273.97|
|  1/1/2022|    Tablet|Electronics| North|       9|9234.0|2523.67|
|  1/1/2022|  Keyboard|Accessories|  East|       6|6372.0| 570.75|
|  1/1/2023|Smartphone|Electronics|  West|      

In [14]:
#Assigning 0's
print("Uncleaned Dataframe Sample (0 Value for Quantity, Sales, OR Profit)")
df_clean1.select(
    "Order_Date", "Name", "Category", "Region", "Quantity", "Sales", "Profit"
).where(
    (col("Quantity") == 0) | (col("Sales") == 0) | (col("Profit") == 0
)).show()

print("Cleaned Dataframe Sample (0 Value for Quantity, Sales, AND Profit)")
df_fclean = df_clean1.withColumn(
    "Quantity",
      when((col("Quantity") == 0) | (col("Sales") == 0) | (col("Profit")==0), 0)
      .otherwise(col("Quantity"))).withColumn(
    "Sales",
       when((col("Quantity") == 0) | (col("Sales") == 0) | (col("Profit")==0), 0)
      .otherwise(col("Sales"))).withColumn(
    "Profit",
       when((col("Quantity") == 0) | (col("Sales") == 0) | (col("Profit")==0), 0)
      .otherwise(col("Sales"))
      )

df_fclean.select(
    "Order_Date", "Name", "Category", "Region", "Quantity", "Sales", "Profit"
).where(
    (col("Quantity") == 0) | (col("Sales") == 0) | (col("Profit") == 0
)).show()



Uncleaned Dataframe Sample (0 Value for Quantity, Sales, OR Profit)
+----------+------+-----------+------+--------+------+------+
|Order_Date|  Name|   Category|Region|Quantity| Sales|Profit|
+----------+------+-----------+------+--------+------+------+
|11/27/2022| Mouse|Accessories|  East|       0|1197.0|106.53|
| 12/1/2023|Camera|Electronics|  West|       1|   0.0|106.35|
+----------+------+-----------+------+--------+------+------+

Cleaned Dataframe Sample (0 Value for Quantity, Sales, AND Profit)
+----------+------+-----------+------+--------+-----+------+
|Order_Date|  Name|   Category|Region|Quantity|Sales|Profit|
+----------+------+-----------+------+--------+-----+------+
|11/27/2022| Mouse|Accessories|  East|       0|  0.0|   0.0|
| 12/1/2023|Camera|Electronics|  West|       0|  0.0|   0.0|
+----------+------+-----------+------+--------+-----+------+



In [8]:
#First Insight
df_fclean.select(
    min("Order_Date").alias("Minimum Date"),
    max("Order_Date").alias("Maximum Date"),
    max("Sales").alias("Maximum Sales"),
    min("Sales").alias("Minimum Sales"),
    avg("Sales").alias("Average Sales"),
    max("Profit").alias("Maximum Profit"),
    min("Profit").alias("Minimum Profit"),
    avg("Profit").alias("Average Profit")
).show()

+------------+------------+-------------+-------------+------------------+--------------+--------------+------------------+
|Minimum Date|Maximum Date|Maximum Sales|Minimum Sales|     Average Sales|Maximum Profit|Minimum Profit|    Average Profit|
+------------+------------+-------------+-------------+------------------+--------------+--------------+------------------+
|    1/1/2022|    9/9/2024|      10782.0|          0.0|3047.4742857142855|       10782.0|           0.0|3047.4742857142855|
+------------+------------+-------------+-------------+------------------+--------------+--------------+------------------+



In [9]:
#Second Insight
region = df_fclean.groupBy("Region").count()
region =region.withColumnRenamed("count", "Total")
region.show()


+------+-----+
|Region|Total|
+------+-----+
| South|  883|
|  East|  861|
|  West|  898|
| North|  858|
+------+-----+



In [10]:
#Third Insight
region_sales = df_fclean.groupBy("Region").agg(
    F.sum("Sales").alias ("Total Sales"),
    F.sum("Profit").alias("Total Profit")
)
region_sales.show()


+------+-----------+------------+
|Region|Total Sales|Total Profit|
+------+-----------+------------+
| South|  2659548.0|   2659548.0|
|  East|  2673913.0|   2673913.0|
|  West|  2843926.0|   2843926.0|
| North|  2488773.0|   2488773.0|
+------+-----------+------------+



In [11]:
#Fourth Insight
df_fclean = df_fclean.withColumn("Order_Date", F.to_date(F.col("Order_Date"), "M/d/yyyy"))
df_fclean = df_fclean.withColumn("Year", F.year(F.col("Order_Date")))

year_sales = df_fclean.groupBy("Year").agg(
    F.sum("Sales").alias("Total Sales"),
    F.sum("Profit").alias("Total Profit")
).orderBy("Year")
year_sales.show()

+----+-----------+------------+
|Year|Total Sales|Total Profit|
+----+-----------+------------+
|2022|  3254773.0|   3254773.0|
|2023|  3786068.0|   3786068.0|
|2024|  3625319.0|   3625319.0|
+----+-----------+------------+



In [12]:
#Fifth Insight
category_sales = df_fclean.groupBy("Category").agg(
    F.sum("Sales").alias("Total Sales"),
    F.sum("Profit").alias("Total Profit")
)
category_sales.show()

+-----------+-----------+------------+
|   Category|Total Sales|Total Profit|
+-----------+-----------+------------+
|     Office|  1094216.0|   1094216.0|
|Electronics|  5325550.0|   5325550.0|
|Accessories|  4246394.0|   4246394.0|
+-----------+-----------+------------+



In [13]:
#Saving in Parquet
parquet_path = "/content/sample_data/parquet"
df_fclean.write.format("parquet").save(parquet_path)
